# Attribution (Saliency + Occlusion + Ablation CAM + Input Ablation) Visualization

Visualizes XAI maps generated by `captum_xai.py`.

Each `.npz` file contains:
- `saliency`         — (3, D, H, W) gradient-based attribution
- `occlusion`        — (3, D, H, W) sliding-window occlusion attribution
- `ablation`         — (1, D, H, W) 3D AblationCAM (class-discriminative, single spatial map)
- `input_ablation`   — (3,) per-input-channel importance weights (score-drop fraction when channel is zeroed)
- `channels`         — ["t2w", "adc", "hbv"]
- `case_id`          — patient/case identifier
- `fold`             — which validation fold

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colorbar import ColorbarBase
from matplotlib.colors import Normalize, LinearSegmentedColormap
from pathlib import Path
from ipywidgets import interact, IntSlider, Dropdown, ToggleButton

# ---------------------------------------------------------------------------
# Model → XAI output directory mapping
# ---------------------------------------------------------------------------
_NB_DIR = Path(".").resolve()   # notebook is in picai_nnunet_semi_supervised_gc_algorithm/
_UMAMBA_XAI = _NB_DIR.parent / "U-MambaMTL-XAI" / "results" / "xai"

MODEL_XAI_DIRS = {
    "nnunet":     _NB_DIR / "results" / "xai",
    "umamba_mtl": _UMAMBA_XAI / "umamba_mtl",
    "swin_unetr": _UMAMBA_XAI / "swin_unetr",
}

MODEL_OPTIONS = [
    ("nnUNet",     "nnunet"),
    ("U-MambaMTL", "umamba_mtl"),
    ("SwinUNETR",  "swin_unetr"),
]

COLORMAP_OPTIONS = [
    ("Hot",     "hot"),
    ("Inferno", "inferno"),
    ("Plasma",  "plasma"),
    ("Turbo",   "turbo"),
    ("Magma",   "magma"),
]

CHANNEL_NAMES_UPPER = ["T2W", "ADC", "HBV"]


def find_attribution_files(model: str = "nnunet"):
    """Find all .npz files produced by captum_xai.py for the given model."""
    xai_dir = MODEL_XAI_DIRS.get(model, MODEL_XAI_DIRS["nnunet"])
    if not xai_dir.exists():
        return []
    return sorted(xai_dir.glob("fold_*/*.npz"))


def load_attribution(npz_path):
    """
    Load attribution data from an npz file.

    Returns a dict with keys:
        saliency        : (3, D, H, W) float32  — or None if not generated
        occlusion       : (3, D, H, W) float32  — or None if not generated
        ablation        : (1, D, H, W) float32  — 3D AblationCAM, or None
        input_ablation  : (n_channels,) float32 — per-channel importance weights, or None
        image           : (3, D, H, W) float32  — preprocessed MRI (same crop)
        prediction      : (1, D, H, W) float32  — cancer probability, or None
        label           : (1, D, H, W) float32  — binary ground-truth, or None
        channels        : list of str  ["t2w", "adc", "hbv"]
        case_id         : str
        fold            : int
    """
    d = np.load(npz_path, allow_pickle=True)

    def _load_optional(key):
        """Load a spatial (ndim > 1) array; returns None for sentinels or missing keys."""
        if key not in d:
            return None
        arr = d[key]
        return arr if arr.ndim > 1 else None  # sentinel is 1-D zeros

    def _load_1d_optional(key):
        """Load a 1-D array (e.g. input_ablation weights); returns None for sentinels."""
        if key not in d:
            return None
        arr = d[key]
        return arr if arr.size > 0 else None  # sentinel is 0-length

    return {
        "saliency":       _load_optional("saliency"),
        "occlusion":      _load_optional("occlusion"),
        "ablation":       _load_optional("ablation"),
        "input_ablation": _load_1d_optional("input_ablation"),
        "image":          d["image"],
        "prediction":     _load_optional("prediction"),
        "label":          _load_optional("label"),
        "channels":       list(d["channels"]),
        "case_id":        str(d["case_id"]),
        "fold":           int(d["fold"]),
    }

def _get_depth(data):
    """Return the depth (D) axis size from the first available data array."""
    for key in ("saliency", "occlusion", "ablation", "image"):
        arr = data.get(key)
        if arr is not None and arr.ndim > 1:
            return arr.shape[1]
    return 1

def normalize_global(arr: np.ndarray) -> np.ndarray:
    """Normalize all channels by the global max → [0, 1]."""
    global_max = arr.max()
    return arr / global_max if global_max > 0 else arr.copy()

def normalize_mri(vol: np.ndarray) -> np.ndarray:
    """Min-max normalize a (D, H, W) MRI volume to [0, 1]."""
    mn, mx = vol.min(), vol.max()
    return (vol - mn) / (mx - mn) if mx > mn else np.zeros_like(vol, dtype=np.float32)

def normalize_input_ablation(weights: np.ndarray) -> np.ndarray:
    """
    Normalize raw score-drop fractions to proportional importance in [0, 1] summing to 1.

    Negative weights (zeroing the channel *increased* the score — possible when the
    zero baseline is not meaningful, e.g. z-score normalized channels) are clipped to 0
    before normalization.  The result represents each channel's share of the total
    positive score drop.
    """
    w_pos = np.clip(weights, 0.0, None)
    total = w_pos.sum()
    return w_pos / total if total > 0 else w_pos.copy()

def overlay(ax, mri_sl, attr_sl, title, cmap="hot", alpha=0.5):
    """Draw MRI in gray with an attribution/probability layer blended on top."""
    ax.imshow(mri_sl, cmap="gray", vmin=0, vmax=1, aspect="equal")
    ax.imshow(attr_sl, cmap=cmap,  vmin=0, vmax=1, aspect="equal", alpha=alpha)
    ax.set_title(title)
    ax.axis("off")

def plot_input_ablation_bar(ax, weights, ch_names, highlight_ch=None,
                             title="Input Ablation\n(channel importance)",
                             normalize=False):
    """
    Draw a bar chart of per-channel input-ablation importance weights.

    weights      : 1-D array of shape (n_channels,)
    ch_names     : list of channel name strings (uppercase)
    highlight_ch : index of the channel to highlight in orange (None = no highlight)
    normalize    : if True, clip negatives to 0 and normalize so weights sum to 1
                   (proportional importance).  Raw score-drop fractions are shown otherwise.
    """
    if normalize:
        display_weights = normalize_input_ablation(weights)
        ylabel = "Norm. importance (∑=1)"
    else:
        display_weights = weights
        ylabel = "Score-drop fraction"

    colors = [
        "tab:orange" if i == highlight_ch else "tab:blue"
        for i in range(len(display_weights))
    ]
    bars = ax.bar(ch_names, display_weights, color=colors, edgecolor="white", linewidth=0.5)

    y_min = min(0.0, float(np.min(display_weights))) * 1.3
    y_max = max(1.05, float(np.max(display_weights)) * 1.2) if len(display_weights) else 1.05
    ax.set_ylim(y_min, y_max)

    span = y_max - y_min
    for bar, w in zip(bars, display_weights):
        va  = "bottom" if w >= 0 else "top"
        yp  = float(w) + span * 0.02 if w >= 0 else float(w) - span * 0.02
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            yp,
            f"{w:.3f}",
            ha="center", va=va, fontsize=8,
        )
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_title(title, fontsize=9)
    ax.set_ylabel(ylabel, fontsize=8)
    ax.tick_params(axis="both", labelsize=8)

def add_legend(fig, attr_cmap="hot", show_zones=False):
    """
    Add a compact legend strip below the figure explaining each overlay colour.

    Four items:
      • Gray gradient    — MRI intensity
      • Attribution cmap — Saliency / Occlusion / Ablation CAM strength
      • Reds gradient    — Predicted cancer probability
      • Solid red patch  — Ground-truth cancer label
    """
    fig.subplots_adjust(bottom=0.18)

    # Helper: draw a tiny horizontal colorbar in a new axes
    def _cbar(left, cmap_name, label):
        ax = fig.add_axes([left, 0.04, 0.18, 0.03])
        cb = ColorbarBase(ax, cmap=plt.get_cmap(cmap_name),
                          norm=Normalize(0, 1), orientation="horizontal")
        cb.set_ticks([0, 1])
        cb.set_ticklabels(["low", "high"])
        ax.set_title(label, fontsize=8, pad=2)

    _cbar(0.08, "gray",     "MRI intensity")
    _cbar(0.30, attr_cmap,  "Attribution strength\n(saliency / occlusion / ablation)")
    _cbar(0.52, "Reds",     "Predicted cancer\nprobability")

    # Binary patch for label
    ax_lbl = fig.add_axes([0.74, 0.04, 0.18, 0.03])
    ax_lbl.set_facecolor((1, 0, 0, 0.6))
    ax_lbl.set_xticks([])
    ax_lbl.set_yticks([])
    ax_lbl.set_title("Ground-truth label\n(cancer region)", fontsize=8, pad=2)

    if show_zones:
        fig.legend(
            handles=[_TZ_PATCH, _PZ_PATCH],
            loc="lower right", fontsize=8, framealpha=0.8,
        )


# ---------------------------------------------------------------------------
# Summary: print available files for all models
# ---------------------------------------------------------------------------
for _label, _model_key in MODEL_OPTIONS:
    _files = find_attribution_files(_model_key)
    print(f"{_label:12s}  ({MODEL_XAI_DIRS[_model_key]})  →  {len(_files)} file(s)")
    for _f in _files:
        print(f"    {_f}")


nnUNet        (/cluster/home/bragehk/master/picai_nnunet_semi_supervised_gc_algorithm/results/xai)  →  422 file(s)
    /cluster/home/bragehk/master/picai_nnunet_semi_supervised_gc_algorithm/results/xai/fold_0/10006_1000006.npz
    /cluster/home/bragehk/master/picai_nnunet_semi_supervised_gc_algorithm/results/xai/fold_0/10022_1000022.npz
    /cluster/home/bragehk/master/picai_nnunet_semi_supervised_gc_algorithm/results/xai/fold_0/10032_1000032.npz
    /cluster/home/bragehk/master/picai_nnunet_semi_supervised_gc_algorithm/results/xai/fold_0/10035_1000035.npz
    /cluster/home/bragehk/master/picai_nnunet_semi_supervised_gc_algorithm/results/xai/fold_0/10038_1000038.npz
    /cluster/home/bragehk/master/picai_nnunet_semi_supervised_gc_algorithm/results/xai/fold_0/10040_1000040.npz
    /cluster/home/bragehk/master/picai_nnunet_semi_supervised_gc_algorithm/results/xai/fold_0/10041_1000041.npz
    /cluster/home/bragehk/master/picai_nnunet_semi_supervised_gc_algorithm/results/xai/fold_0/10059_1

In [2]:
# ─── Zone loading (TZ / PZ) ────────────────────────────────────────────────
# Loads pre-computed zone crops produced by compute_zone_crops.py.
# Run that script first to populate results/xai/zone_crops/.
#
# zone_crop values:  0 = background,  1 = TZ (Transition Zone),  2 = PZ (Peripheral Zone)

_ZONE_CROPS_DIR = _NB_DIR / "results" / "xai" / "zone_crops"
_zone_cache: dict = {}   # str(npz_path) → (D, H, W) int8 or None


def get_zone_crop(npz_path, data=None):
    """
    Load the pre-computed zone crop for *npz_path*.
    Returns (D_crop, H_crop, W_crop) int8 or None if the file is not available.
    """
    key = str(npz_path)
    if key in _zone_cache:
        return _zone_cache[key]
    case_id = npz_path.stem
    fold    = npz_path.parent.name           # "fold_0", "fold_1", …
    zpath   = _ZONE_CROPS_DIR / fold / f"{case_id}.npz"
    if not zpath.exists():
        _zone_cache[key] = None
        return None
    d      = np.load(zpath, allow_pickle=True)
    result = d["zone_crop"]                  # (D_crop, H_crop, W_crop) int8
    _zone_cache[key] = result
    return result


def overlay_zones(ax, zone_slice):
    """Overlay TZ (blue) and PZ (green) as a transparent RGBA layer on *ax*."""
    if zone_slice is None:
        return
    rgba = np.zeros((*zone_slice.shape, 4), dtype=np.float32)
    rgba[zone_slice == 1] = [0.15, 0.35, 1.00, 0.45]   # TZ — blue
    rgba[zone_slice == 2] = [0.10, 0.85, 0.20, 0.45]   # PZ — green
    ax.imshow(rgba, aspect="equal")


_TZ_PATCH = mpatches.Patch(facecolor=(0.15, 0.35, 1.00, 0.7), label="TZ (Transition Zone)")
_PZ_PATCH = mpatches.Patch(facecolor=(0.10, 0.85, 0.20, 0.7), label="PZ (Peripheral Zone)")

_ready = _ZONE_CROPS_DIR.exists()
print(f"Zone crops dir: {_ZONE_CROPS_DIR}  (exists={_ready})")
if not _ready:
    print("  → Run first:  uv run python compute_zone_crops.py")


Zone crops dir: /cluster/home/bragehk/master/picai_nnunet_semi_supervised_gc_algorithm/results/xai/zone_crops  (exists=True)


## Interactive slice viewer — one attribution type at a time

In [ ]:
_model_w1 = Dropdown(options=MODEL_OPTIONS, value="nnunet", description="Model:",
                     style={"description_width": "initial"})

_files1 = find_attribution_files(_model_w1.value)
_npz_w1 = Dropdown(
    options=[(f.stem, f) for f in _files1] if _files1 else [("(no files)", None)],
    description="Case:",
    style={"description_width": "initial"},
    layout={"width": "400px"},
)
_ch_w1    = Dropdown(
    options=[(name, i) for i, name in enumerate(CHANNEL_NAMES_UPPER)],
    description="Channel:",
)
_z_w1     = IntSlider(min=0, max=1, step=1, value=0, description="Slice:")
_alpha_w1 = IntSlider(min=0, max=100, step=5, value=50, description="Overlay α %:")
_cmap_w1  = Dropdown(options=COLORMAP_OPTIONS, value="hot", description="Colormap:")
_zones_w1 = ToggleButton(value=False, description="Show Zones", button_style="info",
                         tooltip="Overlay TZ (blue) and PZ (green) on the MRI panel")

def _update_z1(change):
    if change["new"] is None:
        return
    data = load_attribution(change["new"])
    n = _get_depth(data)
    _z_w1.max = n - 1
    _z_w1.value = min(_z_w1.value, n - 1)

def _update_model1(change):
    files = find_attribution_files(change["new"])
    _npz_w1.options = [(f.stem, f) for f in files] if files else [("(no files)", None)]
    if files:
        _update_z1({"new": _npz_w1.value})

_model_w1.observe(_update_model1, names="value")
_npz_w1.observe(_update_z1, names="value")
_update_z1({"new": _npz_w1.value})  # initialize slice range on load

@interact(model=_model_w1, npz_path=_npz_w1, channel=_ch_w1, z_slice=_z_w1, alpha=_alpha_w1, cmap=_cmap_w1, show_zones=_zones_w1)
def view_single(model, npz_path, channel, z_slice, alpha, cmap, show_zones):
    if npz_path is None:
        print(f"No attribution files found for model '{model}'.")
        return
    data     = load_attribution(npz_path)
    n_slices = _get_depth(data)
    z        = z_slice
    mri      = normalize_mri(data["image"][channel])
    ch_name  = data["channels"][channel].upper()
    ch_names_upper = [c.upper() for c in data["channels"]]
    a        = alpha / 100

    has_saliency       = data["saliency"] is not None
    has_occlusion      = data["occlusion"] is not None
    has_ablation       = data["ablation"] is not None
    has_input_ablation = data["input_ablation"] is not None
    has_pred           = data["prediction"] is not None
    has_label          = data["label"] is not None
    n_panels = (1
                + int(has_saliency)
                + int(has_occlusion)
                + int(has_ablation)
                + int(has_input_ablation)
                + int(has_pred)
                + int(has_label))

    fig, axes = plt.subplots(1, n_panels, figsize=(5 * n_panels, 5))
    axes = list(axes) if n_panels > 1 else [axes]
    mri_sl = mri[z]

    axes[0].imshow(mri_sl, cmap="gray", vmin=0, vmax=1, aspect="equal")
    axes[0].set_title(f"MRI  {ch_name}  z={z}/{n_slices-1}")
    axes[0].axis("off")

    # Optionally overlay TZ/PZ zones on the MRI panel
    if show_zones:
        zone_crop = get_zone_crop(npz_path, data)
        if zone_crop is not None and z < zone_crop.shape[0]:
            print("max: ", zone_crop.max())
            print("min: ", zone_crop.min())
            overlay_zones(axes[0], zone_crop[z])
        else:
            if zone_crop is None:
                print("Zone crop is None!")
            else:
                print("Bruh")

    idx = 1
    if has_saliency:
        sal = normalize_global(data["saliency"])
        overlay(axes[idx], mri_sl, sal[channel, z], f"Saliency  {ch_name}", cmap=cmap, alpha=a)
        idx += 1
    if has_occlusion:
        occ = normalize_global(data["occlusion"])
        overlay(axes[idx], mri_sl, occ[channel, z], f"Occlusion  {ch_name}", cmap=cmap, alpha=a)
        idx += 1
    if has_ablation:
        abl = normalize_global(data["ablation"])
        overlay(axes[idx], mri_sl, abl[0, z], f"Ablation CAM  {ch_name}", cmap=cmap, alpha=a)
        idx += 1
    if has_input_ablation:
        plot_input_ablation_bar(
            axes[idx], data["input_ablation"], ch_names_upper,
            highlight_ch=channel,
            title=f"Input Ablation\n(↑ = {ch_name} highlighted)",
        )
        idx += 1
    if has_pred:
        overlay(axes[idx], mri_sl, data["prediction"][0, z],
                f"Prediction  {ch_name}", cmap="Reds", alpha=a)
        idx += 1
    if has_label:
        overlay(axes[idx], mri_sl, data["label"][0, z],
                f"Label  {ch_name}", cmap="Reds", alpha=min(a + 0.2, 1.0))

    plt.suptitle(f"[{model}]  case {data['case_id']}  fold {data['fold']}", fontsize=12)
    add_legend(fig, cmap, show_zones=show_zones)
    plt.show()

# 10268 has 3 lesions, nnUNet preds 2

interactive(children=(Dropdown(description='Model:', options=(('nnUNet', 'nnunet'), ('U-MambaMTL', 'umamba_mtl…

In [4]:
_model_w2 = Dropdown(options=MODEL_OPTIONS, value="nnunet", description="Model:",
                     style={"description_width": "initial"})

_files2 = find_attribution_files(_model_w2.value)
_npz_w2 = Dropdown(
    options=[(f.stem, f) for f in _files2] if _files2 else [("(no files)", None)],
    description="Case:",
    style={"description_width": "initial"},
    layout={"width": "400px"},
)
_z_w2     = IntSlider(min=0, max=1, step=1, value=0, description="Slice:")
_alpha_w2 = IntSlider(min=0, max=100, step=5, value=50, description="Overlay α %:")
_cmap_w2  = Dropdown(options=COLORMAP_OPTIONS, value="hot", description="Colormap:")
_zones_w2 = ToggleButton(value=False, description="Show Zones", button_style="info",
                         tooltip="Overlay TZ (blue) and PZ (green) on each MRI column")

def _update_z2(change):
    if change["new"] is None:
        return
    data = load_attribution(change["new"])
    n = _get_depth(data)
    _z_w2.max = n - 1
    _z_w2.value = min(_z_w2.value, n - 1)

def _update_model2(change):
    files = find_attribution_files(change["new"])
    _npz_w2.options = [(f.stem, f) for f in files] if files else [("(no files)", None)]
    if files:
        _update_z2({"new": _npz_w2.value})

_model_w2.observe(_update_model2, names="value")
_npz_w2.observe(_update_z2, names="value")
_update_z2({"new": _npz_w2.value})  # initialize slice range on load

@interact(model=_model_w2, npz_path=_npz_w2, z_slice=_z_w2, alpha=_alpha_w2, cmap=_cmap_w2, show_zones=_zones_w2)
def view_all_channels(model, npz_path, z_slice, alpha, cmap, show_zones):
    if npz_path is None:
        print(f"No attribution files found for model '{model}'.")
        return
    data      = load_attribution(npz_path)
    ch_names  = [c.upper() for c in data["channels"]]
    n_ch      = len(ch_names)
    n_slices  = _get_depth(data)
    z         = z_slice
    a         = alpha / 100

    has_saliency       = data["saliency"] is not None
    has_occlusion      = data["occlusion"] is not None
    has_ablation       = data["ablation"] is not None
    has_input_ablation = data["input_ablation"] is not None
    has_pred           = data["prediction"] is not None
    has_label          = data["label"] is not None

    row_labels = ["MRI"]
    if has_saliency:       row_labels.append("Saliency")
    if has_occlusion:      row_labels.append("Occlusion")
    if has_ablation:       row_labels.append("Ablation CAM")
    if has_input_ablation: row_labels.append("Input Ablation")
    if has_pred:           row_labels.append("Prediction")
    if has_label:          row_labels.append("Label")
    n_rows = len(row_labels)

    fig, axes = plt.subplots(n_rows, n_ch, figsize=(5 * n_ch, 5 * n_rows), squeeze=False)

    abl = normalize_global(data["ablation"]) if has_ablation else None

    # Load zone crop once for this npz (cached after first call)
    zone_crop = get_zone_crop(npz_path, data) if show_zones else None

    for col, ch_name in enumerate(ch_names):
        mri = normalize_mri(data["image"][col])
        sl  = mri[z]

        axes[0, col].imshow(sl, cmap="gray", vmin=0, vmax=1, aspect="equal")
        axes[0, col].set_title(f"MRI  {ch_name}  z={z}/{n_slices-1}")
        axes[0, col].axis("off")

        # Overlay TZ/PZ on the MRI row
        if show_zones and zone_crop is not None and z < zone_crop.shape[0]:
            overlay_zones(axes[0, col], zone_crop[z])

        row = 1
        if has_saliency:
            sal = normalize_global(data["saliency"])
            overlay(axes[row, col], sl, sal[col, z], f"Saliency  {ch_name}", cmap=cmap, alpha=a)
            row += 1
        if has_occlusion:
            occ = normalize_global(data["occlusion"])
            overlay(axes[row, col], sl, occ[col, z], f"Occlusion  {ch_name}", cmap=cmap, alpha=a)
            row += 1
        if has_ablation:
            overlay(axes[row, col], sl, abl[0, z],
                    f"Ablation CAM  {ch_name}", cmap=cmap, alpha=a)
            row += 1
        if has_input_ablation:
            plot_input_ablation_bar(
                axes[row, col],
                data["input_ablation"],
                ch_names,
                highlight_ch=col,
                title=f"Input Ablation  {ch_name}",
            )
            row += 1
        if has_pred:
            overlay(axes[row, col], sl, data["prediction"][0, z],
                    f"Prediction  {ch_name}", cmap="Reds", alpha=a)
            row += 1
        if has_label:
            overlay(axes[row, col], sl, data["label"][0, z],
                    f"Label  {ch_name}", cmap="Reds", alpha=min(a + 0.2, 1.0))

    for row, label in enumerate(row_labels):
        axes[row, 0].set_ylabel(label, fontsize=13, labelpad=8)

    plt.suptitle(f"[{model}]  case {data['case_id']}  fold {data['fold']}", fontsize=13)
    add_legend(fig, cmap, show_zones=show_zones)
    plt.show()


interactive(children=(Dropdown(description='Model:', options=(('nnUNet', 'nnunet'), ('U-MambaMTL', 'umamba_mtl…

_model_w3 = Dropdown(options=MODEL_OPTIONS, value="nnunet", description="Model:",
                     style={"description_width": "initial"})

_files3 = find_attribution_files(_model_w3.value)
_npz_w3 = Dropdown(
    options=[(f.stem, f) for f in _files3] if _files3 else [("(no files)", None)],
    description="Case:",
    style={"description_width": "initial"},
    layout={"width": "400px"},
)
_alpha_w3 = IntSlider(min=0, max=100, step=5, value=50, description="Overlay α %:")
_cmap_w3  = Dropdown(options=COLORMAP_OPTIONS, value="hot", description="Colormap:")
_zones_w3 = ToggleButton(value=False, description="Show Zones", button_style="info",
                         tooltip="Overlay TZ (blue) and PZ (green) MIP on each MRI-MIP panel")

def _update_model3(change):
    files = find_attribution_files(change["new"])
    _npz_w3.options = [(f.stem, f) for f in files] if files else [("(no files)", None)]

_model_w3.observe(_update_model3, names="value")

@interact(model=_model_w3, npz_path=_npz_w3, alpha=_alpha_w3, cmap=_cmap_w3, show_zones=_zones_w3)
def view_mip(model, npz_path, alpha, cmap, show_zones):
    if npz_path is None:
        print(f"No attribution files found for model '{model}'.")
        return
    data      = load_attribution(npz_path)
    ch_names  = [c.upper() for c in data["channels"]]
    n_ch      = len(ch_names)
    n_slices  = _get_depth(data)
    a         = alpha / 100

    has_saliency       = data["saliency"] is not None
    has_occlusion      = data["occlusion"] is not None
    has_ablation       = data["ablation"] is not None
    has_input_ablation = data["input_ablation"] is not None
    has_pred           = data["prediction"] is not None
    has_label          = data["label"] is not None

    row_labels = ["MRI MIP"]
    if has_saliency:       row_labels.append("Saliency MIP")
    if has_occlusion:      row_labels.append("Occlusion MIP")
    if has_ablation:       row_labels.append("Ablation CAM MIP")
    if has_input_ablation: row_labels.append("Input Ablation")
    if has_pred:           row_labels.append("Prediction MIP")
    if has_label:          row_labels.append("Label MIP")
    n_rows = len(row_labels)

    fig, axes = plt.subplots(n_rows, n_ch, figsize=(5 * n_ch, 5 * n_rows), squeeze=False)

    abl = normalize_global(data["ablation"]) if has_ablation else None

    # Load zone crop once for this npz (cached after first call)
    zone_crop = get_zone_crop(npz_path, data) if show_zones else None
    zone_mip  = zone_crop.max(axis=0) if zone_crop is not None else None

    for col, ch_name in enumerate(ch_names):
        mri_mip = normalize_mri(data["image"][col]).max(axis=0)

        axes[0, col].imshow(mri_mip, cmap="gray", vmin=0, vmax=1, aspect="equal")
        axes[0, col].set_title(f"MRI MIP  {ch_name}  ({n_slices} slices)")
        axes[0, col].axis("off")

        # Overlay TZ/PZ maximum-intensity projection on the MRI-MIP row
        if show_zones and zone_mip is not None:
            overlay_zones(axes[0, col], zone_mip)

        row = 1
        if has_saliency:
            sal = normalize_global(data["saliency"])
            overlay(axes[row, col], mri_mip, sal[col].max(axis=0),
                    f"Saliency MIP  {ch_name}", cmap=cmap, alpha=a)
            row += 1
        if has_occlusion:
            occ = normalize_global(data["occlusion"])
            overlay(axes[row, col], mri_mip, occ[col].max(axis=0),
                    f"Occlusion MIP  {ch_name}", cmap=cmap, alpha=a)
            row += 1
        if has_ablation:
            abl_mip = abl[0].max(axis=0)
            overlay(axes[row, col], mri_mip, abl_mip,
                    f"Ablation CAM MIP  {ch_name}", cmap=cmap, alpha=a)
            row += 1
        if has_input_ablation:
            plot_input_ablation_bar(
                axes[row, col],
                data["input_ablation"],
                ch_names,
                highlight_ch=col,
                title=f"Input Ablation  {ch_name}",
            )
            row += 1
        if has_pred:
            pred_mip = data["prediction"][0].max(axis=0)
            overlay(axes[row, col], mri_mip, pred_mip,
                    f"Prediction MIP  {ch_name}", cmap="Reds", alpha=a)
            row += 1
        if has_label:
            lbl_mip = data["label"][0].max(axis=0)
            overlay(axes[row, col], mri_mip, lbl_mip,
                    f"Label MIP  {ch_name}", cmap="Reds", alpha=min(a + 0.2, 1.0))

    for row, label in enumerate(row_labels):
        axes[row, 0].set_ylabel(label, fontsize=13, labelpad=8)

    plt.suptitle(f"[{model}]  case {data['case_id']}  fold {data['fold']}", fontsize=13)
    add_legend(fig, cmap, show_zones=show_zones)
    plt.show()
